# Observable neural continued-learning pilot
**RTX 2070 / next-agent handoff:** see [GPU session instructions](../docs/RTX2070-HANDOFF.md). This implementation and the neural dependency extra are currently CPU-only; a GPU machine does not automatically enable CUDA.
**Proposed research protocol, not a team-final benchmark or a reproduction.** Default Run All is an **invented Amazon-schema fixture** on CPU. Set `CP4285_NEURAL_MODE=real` and explicit `CP4285_AMAZON_A` / `CP4285_AMAZON_B` paths only for an authorized bounded local run. `audit` prepares and validates local data but does not train. No downloads, GPU or expensive sweeps. Saved fixture numbers are software evidence, never Amazon results.

Question: after initial A next-review learning, how do fixed A retention and B adaptation rankings change under frozen, A-only, 50% B and B-only continuation? Improvement, stability and decline are all valid. This is a feasibility pilot, not evidence of catastrophic forgetting or neural superiority.

Sources: [original SASRec (Kang & McAuley, 2018)](https://arxiv.org/abs/1808.09781); [official Amazon Reviews 2023 schema](https://amazon-reviews-2023.github.io/data_loading/jsonl.html). Original SASRec uses causal self-attention and sampled binary cross-entropy. Here: a small pre-norm PyTorch Transformer, GELU, shared positions/encoder, separate domain-specific tied item tables, **full-domain softmax CE only at the final history position**, input dropout only. These are declared deviations, not a reproduction claim.


## 1. Predeclared bounded protocol and limits
- All ratings: predict the next **reviewed** parent product, not purchase, preference or ideal recommendation. Preserve source user/product IDs, domain namespaces and file-row event IDs.
- Read at most 50,000 rows per category; retain up to 32 lexical users with at least five distinct-timestamp events. File-prefix sampling and activity eligibility are biased offline choices, not representative sampling or calendar-aligned distribution shift. The bounded prefix can truncate user histories.
- Sort each user's events by timestamp; keep first source row at tied timestamps, retain repeated products otherwise. First event is context; next two targets are initial A, next two are fresh continuation. B's first three events are context only, never initial gradients. Immediately following event is held out. Later events are unused.
- SHA256(user) parity deterministically partitions development/test users. Each user supplies at most one held-out next-review label, so validation events never become test prefixes. Different-domain histories are never spliced, even for shared users.
- Catalogues are frozen from selected pre-holdout events, including the declared incoming training inventory. This is a known-catalogue offline experiment, **not strict online item discovery**. Exclude and count unknown-item holdouts; never expand vocabulary using held-out labels.
- Initial A: two epochs, choose first best development-A NDCG@1. Development-B is a diagnostic, **not** selection. Fixed retention-A/development-B/adaptation-B queries, labels, full prepared catalogues and lexical ties at every checkpoint; no seen-item masking. Single-target Recall equals Hit.
- Two blocks of four targets each for A-only/mixed/B-only; B share is B/(A+B). No stream recycling. At equal total exposure, mixed means less A: this does not isolate harmful B gradients. Concatenate A then B within each block; order is a disclosed confound.
- Every supplied target gets one Adam step, last 10 context items, clipping at 1.0. Reset Adam once at continuation; preserve it thereafter. Branch weights, optimizer and input-dropout RNG are independent. Fixed seed 7, width 16, one layer/two heads, learning rate .003, dropout .1. No tuning on retention/adaptation curves.
- One small seed/width is a software and feasibility check. Larger capacities, seed aggregation, representative cohorts and a globally temporal protocol require a separately agreed budget. No claim that this budget converges.


In [ ]:
import csv
import json
import os
import platform
import tempfile
import time
from dataclasses import replace
from pathlib import Path
from uuid import uuid4

import torch
from cp4285.experiment import plot_results, run
from cp4285.experiment.amazon import AmazonSource
from cp4285.experiment.artifacts import save_json
from cp4285.models.sasrec import SASRecRecommender

MODE = os.environ.get('CP4285_NEURAL_MODE', 'fixture')
assert MODE in {'fixture', 'audit', 'real'}
torch.set_num_threads(1)  # Bound tiny CPU work; no GPU.
started = time.perf_counter()
print({'mode': MODE, 'python': platform.python_version(), 'torch': torch.__version__, 'threads': torch.get_num_threads()})


In [ ]:
if MODE == 'fixture':
    fixture_root = Path(tempfile.mkdtemp(prefix='cp4285-schema-fixture-'))
    paths = {}
    for domain in ('A', 'B'):
        path = fixture_root / f'{domain}.csv'
        with path.open('w', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=['user_id', 'parent_asin', 'rating', 'timestamp'])
            writer.writeheader()
            for user in range(16):
                for index in range(7):
                    writer.writerow(dict(user_id=f'fixture-user-{user}', parent_asin=f'fixture-item-{index % 3}', rating=1 + index % 5, timestamp=1000 * index + user))
        paths[domain] = path
else:
    paths = {d: Path(os.environ[f'CP4285_AMAZON_{d}']) for d in ('A', 'B')}

source = AmazonSource(paths, max_rows=50000, max_users=32, initial_targets=2, update_targets=2, block_size=4, blocks=2)
data = source.load_data()  # Fails, rather than expanding bounds, if cohort/budget is inadequate.
if MODE == 'fixture':
    data = replace(data, provenance='INVENTED AMAZON-SCHEMA FIXTURE; NOT AMAZON OBSERVATIONS; ' + json.dumps(source.audit, sort_keys=True))
data.validate()
print('DATA AUDIT (aggregates only; private original IDs remain in ignored prepared artifact):')
print(json.dumps(source.audit, indent=2))
print({'initial_A_targets': len(data.initial), 'tasks': {t.name: len(t.examples) for t in (data.validation, *data.tests)}, 'audit_seconds': round(time.perf_counter() - started, 3)})


## 2. Audit gate, model and observable execution
Check file names, scanned-prefix hashes (not full-file hashes), source timestamp ranges/units, excluded ties and unknown labels, available fresh targets and development/test counts above. Official Amazon 2023 timestamps are milliseconds; the loader preserves integer source units without guessing conversion. Small CSV exports must keep these same ID columns. No metadata/text is needed. Unknown labels or insufficient fresh targets must not trigger silent resampling or protocol retuning.

`run` owns selection, branches, metrics and immutable prepared inputs. The observer only prints phase/elapsed seconds/mean target loss, saves progress JSON and create-only complete checkpoints. It does not alter learning or choose models from tests. Full checkpoint includes weights, Adam moments, RNG and counters. Runner JSON plus source fingerprints and model/Torch metadata preserve provenance; partial failures remain visible and require a new output path. All user-level artifacts stay under ignored `runs/`.


In [ ]:
output = Path('runs') / f'neural-{MODE}-{uuid4().hex[:12]}'
model = SASRecRecommender(source.vocabularies, width=16, heads=2, layers=1, max_history=10, learning_rate=0.003, dropout=0.1, seed=7)
print(json.dumps(model.describe(), indent=2))
progress = []

def observe(event, learner):
    row = {**event, 'elapsed_seconds': round(time.perf_counter() - started, 4), 'mean_target_loss': learner.last_loss, 'optimizer_targets_total': learner.targets_seen}
    progress.append(row)
    save_json(output / 'progress' / f'{len(progress):04d}.json', row)
    if event['phase'] == 'selected':
        learner.save(output / 'checkpoints' / 'selected.pt')
    if event['phase'] == 'evaluation' and event['task'] == 'retention-A':
        learner.save(output / 'checkpoints' / f"{event['condition']}-step-{event['step']}.pt")
    print(json.dumps(row, sort_keys=True), flush=True)

result = None
if MODE != 'audit':
    result = run(model, data, output=output, initial_epochs=2, k=1, observe=observe)
    save_json(output / 'audit.json', source.audit)
    print('Saved create-only artifacts:', output, 'selected epoch:', result['selected_epoch'])
else:
    print('AUDIT ONLY: no training, test metrics or checkpoints produced.')


## 3. Per-checkpoint retention/adaptation and B development learning
Read raw records before graphs. The frozen branch should remain unchanged. B development is distinct from adaptation-B test; report what it actually does. A non-improving B development score or unstable/large losses means learning at this budget is **not established**, not a reason to tune using test curves. A decrease in A is an outcome under this protocol, not proof that memory was erased. A flat score may reflect tiny candidate sets, one seed or insufficient learning.


In [ ]:
if result is not None:
    saved = json.loads((output / 'results.json').read_text())
    print('VALIDATION TRACE:', saved['validation'])
    print('condition | step | task | n | candidates | NDCG@1 | Hit@1 | change | exposures')
    for row in saved['records']:
        print(f"{row['condition']:7} | {row['step']} | {row['task']:13} | {row['n']} | {row['candidate_count']} | {row['ndcg']:.4f} | {row['hit']:.4f} | {row['ndcg_change_from_initial']:+.4f} | {row['targets_seen']}")
    bdev = [r for r in saved['records'] if r['condition'] == 'B-only' and r['task'] == 'development-B']
    print('B development NDCG change:', bdev[-1]['ndcg'] - bdev[0]['ndcg'], '(diagnostic, no tuning; improvement is not guaranteed)')
    for task in ('retention-A', 'adaptation-B'):
        svg = plot_results(output / 'results.json', task=task)
        with (output / f'{task}.svg').open('x') as stream:
            stream.write(svg)
        try:
            from IPython.display import SVG, display
            display(SVG(svg))
        except ImportError:
            print('Saved graph:', output / f'{task}.svg')


## 4. Reload and independent next-update check
This last check is software validation on disposable copies, **not extra experiment exposure**. It reuses one target solely to assert that reloaded Adam/RNG state produces exactly the same next update. Main run budgets/metrics above are already saved and never modified. Only load trusted local checkpoints; publication refuses overwrites. Automated tests separately cover score-state purity, causal masking, candidate order, full-state forks and a B-only route into shared parameters while A item weights remain untouched.


In [ ]:
if result is not None:
    restored = SASRecRecommender.load(output / 'checkpoints' / 'B-only-step-2.pt')
    clone = restored.fork()
    restored.train(data.initial[:1])
    clone.train(data.initial[:1])
    assert all(torch.equal(value, clone.network.state_dict()[key]) for key, value in restored.network.state_dict().items())
    assert torch.equal(restored.generator.get_state(), clone.generator.get_state())
    print('Reload/fork next-update equivalence: PASS; caller remained untrained:', model.targets_seen == 0)
    print('Evidence label:', 'FIXTURE ONLY — NOT REAL AMAZON' if MODE == 'fixture' else 'BOUNDED LOCAL AMAZON PILOT — NOT A CONCLUSIVE STUDY')
    print('Total elapsed seconds:', round(time.perf_counter() - started, 3))


## Interpretation and next decision
This notebook provides inspectable software and bounded feasibility evidence. It cannot establish a capacity trend, generalization across seeds, causal attribution to B gradients, representative Amazon performance or a learning-vs-forgetting trade-off from one tiny run. The held-out fixed-anchor queries contain permitted future update context; their reuse is deliberate and frozen, not a calendar-causal deployment simulation. A second model's benchmark must use the same prepared inputs and evaluator. Agree a representative sampling/time protocol, candidate policy, development learning criterion and compute budget before extending this pilot. The canonical Google Slides deck is unchanged.
